# **Data Mining Project: ACDC' Customer Segmentation (Group 30)**
This first notebook will be about Exploratory Data Analysis (EDA). Here it will be examined the coworking membership dataset in detail, focusing on uncovering meaningful patterns, identifying limitations, and generating initial hypotheses about
customer groups. It reads `acdc_dataset.csv`from `../data/raw/`, and saves the submission in `../data/processed/`.

## **Part 1. Exploratory Data Analysis**

### **1.Setup**

In [8]:
# library imports

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

#created paths 

raw_data_path=os.path.join("..", "data", "raw", "acdc_dataset.csv")
processed_data_path=os.path.join("..", "data", "processed")

#created dataframe

df_acdcd_data= pd.read_csv(raw_data_path, index_col="member_id", sep=";")

### **2.Loading Data and First Look**

#### **2.1. Overall First Look**

In [12]:
print(f"Rows:{df_acdcd_data.shape[0]:,} | Columns:{df_acdcd_data.shape[1]:,}")
df_acdcd_data.head()

Rows:15,465 | Columns:30


,age,gender,industry,country,is_team_account,n_colleagues_on_account,plan_tier,plan_days_per_week,membership_start,membership_end,...,uses_event_space,uses_guest_passes,uses_print,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker,n_renewals,lifetime_value
member_id,,,,,,,,,,,,,,,,,,,,,
CW000001,38.0,Female,Information Technology,Portugal,False,0,flex,3,2021-08-01,2023-03-06,...,False,True,0,False,True,True,False,False,2,4477.97
CW000002,38.0,Male,Information Technology,Portugal,True,2,HOT_DESK,2,2023-11-24,1970-01-01,...,False,False,0,True,False,False,False,False,0,165.86
CW000003,43.0,Male,Education,DE,True,1,flex,3,2021-12-03,1970-01-01,...,False,False,0,False,False,False,False,False,1,4486.17
CW000004,38.0,F,Legal,portugal,False,0,flex,3,2022-01-02,2022-02-15,...,False,False,0,False,False,False,False,False,0,258.63
CW000005,45.0,Female,Information Technology,United Kingdom,False,0,hot_desk,2,2023-10-24,1970-01-01,...,False,False,0,False,False,True,False,False,0,335.74


A preliminary examination of the first few records reveals immediate formatting inconsistencies across categorical variables. 
Specifically, the `gender`, `country`, and `plan_tier` features exhibit non-standardised string entries (e.g., variations in casing, whitespace, and abbreviations). 

Furthermore, a relational check between temporal features indicates logical contradictions, as certain entries display a `membership_start` date that post-dates the corresponding `membership_end` date.

#### **Metadata**
- *member_id*: Unique identifier of each member

- *age*: Age of the member at the extraction date.

- *gender*: Gender of the member.

- *country*: The country where the member (or their associated company) is based.

- *industry*: The industry sector the member works in.

- *is_team_account*: Whether the membership is held by a team/company account rather than an individual.

- *n_colleagues_on_account*: Number of colleagues sharing the account.

- *plan_tier*: Current membership tier. (hot_desk < flex < dedicated_desk <full_time < private_office)

- *plan_days_per_week*: Number of days per week the member's plan entitles them to use the space.

- *membership_start*: Date the membership first began.

- *membership_end*: Date the membership first ended.

- *last_badge_swipe*: Date of the member's most recent physical entry to the space.

- *days_since_last_visit*:  Number of days since the member's last recorded visit.

- *total_visits*: Total number of visits the member made over the whole membership.

- *avg_weekly_visits*: Average number of visits per week over the membership.

- *entitled_visits_last_period*: Number of visits the member's plan entitled them to during the last period of their membership.

- *actual_visits_last_period*: Number of visits the member actually made during the last period of their membership.
- *meeting_room_hours*: Hours of meeting-room usage recorded for the member.

- *event_space_hours*: Hours of event-space usage recorded for the member.

- *guest_passes_issued*: Number of guest passes the member issued.

- *uses_meeting_rooms*: Whether the member ever used the meeting-room add-on.

- *uses_event_space*: Whether the member ever used the event-space add-on.

- *uses_guest_passes*: Whether the member ever used the guest-pass add-on.

- *uses_print*: Whether the member used the print add-on.

- *uses_mail_handling*: Whether the member used the mail-handling add-on.

- *uses_phone_booths*: Whether the member used the phone-booth add-on.

- *uses_day_passes*: Whether the member used the day-pass add-on.

- *uses_storage*: Whether the member used the storage add-on.

- *uses_bike_locker*: Whether the member used the bike-locker add-on.

- *n_renewals*: Number of times the membership was renewed.

- *lifetime_value*: Total revenue attributed to the member over the whole membership.

In [10]:
df_acdcd_data.info()
print()
df_acdcd_data.dtypes()

<class 'pandas.core.frame.DataFrame'>
Index: 15465 entries, CW000001 to CW015101
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          14907 non-null  float64
 1   gender                       14935 non-null  object 
 2   industry                     15465 non-null  object 
 3   country                      15035 non-null  object 
 4   is_team_account              15465 non-null  bool   
 5   n_colleagues_on_account      15465 non-null  int64  
 6   plan_tier                    15360 non-null  object 
 7   plan_days_per_week           15465 non-null  int64  
 8   membership_start             15465 non-null  object 
 9   membership_end               15465 non-null  object 
 10  last_badge_swipe             15465 non-null  object 
 11  days_since_last_visit        15465 non-null  int64  
 12  total_visits                 15465 non-null  int64  
 13  avg_weekly_

TypeError: 'Series' object is not callable

The dataset contains incomplete entries across several columns, most notably `age`, `gender`, `country`, `plan_tier`, `meeting_room_hours`, and `guest_passes_issued`. Additionally, memory usage, feature data types, and total row counts are established to guide subsequent data type casting and handling of missing values.

In [14]:
df_acdcd_data.describe().T

,count,mean,std,min,25%,50%,75%,max
age,14907.0,34.858724,10.394649,1.0,29.00,34.00,40.00,149.00
n_colleagues_on_account,15465.0,0.537019,1.177668,-1.0,0.00,0.00,1.00,9.00
plan_days_per_week,15465.0,3.387262,1.458818,0.0,2.00,3.00,5.00,9.00
days_since_last_visit,15465.0,117.109279,275.532807,-956.0,4.00,13.00,55.00,1212.00
total_visits,15465.0,110.406531,182.939340,-112.0,8.00,34.00,132.00,6530.00
avg_weekly_visits,15465.0,1.777214,1.937515,0.0,0.71,1.42,2.41,47.40
entitled_visits_last_period,15465.0,27.517393,13.480235,0.0,17.39,26.09,43.48,78.27
actual_visits_last_period,15465.0,13.800970,11.387207,0.0,5.00,10.00,20.00,60.00
meeting_room_hours,14167.0,33.075570,428.169678,-56.6,0.00,0.00,7.20,41220.00
event_space_hours,15465.0,5.117220,26.013097,-13.5,0.00,0.00,0.00,602.80


Descriptive statistics highlight notable distributional anomalies across numerical features. The `age` feature displays an implausible maximum value of 149 years, indicating data entry errors. 

Additionally, several numerical variables—including `n_colleagues_on_account`, `days_since_last_visit`, `total_visits`, `meeting_room_hours`, `event_space_hours`, `guest_passes_issued` and `n_renewals`, exhibit impossible negative minimum values. 

Lastly, a comparison between the 75th percentile and maximum values for `n_colleagues_on_account`, `avg_weekly_visits`, `meeting_room_hours`, `event_space_hours`, and `guest_passes_issued` reveals a severe right-skewed distribution, suggesting the presence of extreme upper values that warrant dedicated outlier analysis.

#### **2.2. Grouping the Variables**

To better understand, visualise and organize our data we chose to group the variables by themes: